'''
Source: Impact4cast (Max Planck Institute)
Original code from the Max Planck Institute for Informatics / Max Planck Institute for Security and Privacy research group.

Modifications: Bug fixes for checkpoint resume mechanism and dataset adaptation for scientific entity impact prediction.
'''



### 这个Notebook的主要目的是为图神经网络/链接预测任务准备训练数据，特别是构建未连接节点对(unconnected pairs)及其随时间变化的解决方案(solution)。代码按照时间窗口(2016→2019→2022)分批处理大规模图数据。

## prepare unconnected pairs and its solution year_delta=3

In [ ]:
import os
import sys
import json
import pickle
import gzip
from datetime import datetime, date
import random, time
import numpy as np
from scipy import sparse
import networkx as nx
from collections import defaultdict,Counter
import itertools
import copy
from itertools import combinations
import pandas as pd

### read full graph
#### ['v1', 'v2', 'time', 'ct', 'c2025','c2024;,'c2023', 'c2022', 'c2021', 'c2020', 'c2019', 'c2018', 'c2017', 'c2016', 'c2015', 'c2014', 'c2013', 'c2012'])

读取完整的动态图数据，包含v1、v2、时间戳和各年度引用计数,从parquet文件读取完整的动态图数据(full_dynamic_graph.parquet)
数据量：1,170,455条边

In [ ]:
time_start = time.time()

graph_folder="E:\\study\\research\\impact4cast_cscl\\data_concept_graph"
graph_file=os.path.join(graph_folder,"full_dynamic_graph.parquet")

full_graph = pd.read_parquet(graph_file)
print(f"Done, read full_graph: {len(full_graph)}; elapsed_time: {time.time() - time_start} seconds")

检查数据是否满足v1 < v2的排序条件（确保无向边的标准化存储）

In [ ]:
time_start=time.time()
is_smaller = np.all(full_graph['v1'] < full_graph['v2'])
print(f"is_smaller: {is_smaller}; elapsed_time: {time.time()-time_start}\n")

### get the unconnected-pairs years_delta=3

In [ ]:
#定义关键时间点：2016、2019、2022年底相对于1990-01-01的天数
NUM_OF_VERTICES=38739 ## number of concepts

vertex_degree_cutoff=1
min_edges=1
years_delta=3

day_origin = date(1990,1,1)
day_2016 = (date(2016, 12, 31)- day_origin).days
day_2019 = (date(2019, 12, 31)- day_origin).days
day_2022 = (date(2022, 12, 31)- day_origin).days
print(f"day_2016: {day_2016}; day_2019: {day_2019}; day_2022: {day_2022}\n")

#### get full_graph up to 2016,2019,2022

分别获取2016、2019、2022年底前的子图数据

In [ ]:
print(f"full_dynamic_graph: {len(full_graph)}")
time_start = time.time()
full_graph_2016=full_graph[full_graph['time']<=day_2016]
print(f"full_graph_2016: {len(full_graph_2016)}; elapsed_time: {time.time()-time_start}")


time_start = time.time()
full_graph_2019=full_graph[full_graph['time']<=day_2019]
print(f"full_graph_2019: {len(full_graph_2019)}; elapsed_time: {time.time()-time_start}")


time_start = time.time()
full_graph_2022=full_graph[full_graph['time']<=day_2022]
print(f"full_graph_2022: {len(full_graph_2022)}; elapsed_time: {time.time()-time_start}")

#### get all the vertex pairs

从各时间点的图中提取所有存在的边对(v1, v2)，并去重

In [ ]:
time_start=time.time()
pairs_2016 = full_graph_2016[['v1', 'v2']].drop_duplicates()
print(f"pairs_2016: {len(pairs_2016)}; elapsed_time: {time.time()-time_start}")

time_start=time.time()
pairs_2019 = full_graph_2019[['v1', 'v2']].drop_duplicates()
print(f"pairs_2019: {len(pairs_2019)}; elapsed_time: {time.time()-time_start}")

time_start=time.time()
pairs_2022 = full_graph_2022[['v1', 'v2']].drop_duplicates()
print(f"pairs_2022: {len(pairs_2022)}; elapsed_time: {time.time()-time_start}")

#### get all-combine-pairs while degree >= vertex_degree_cutoff

计算每个节点的度数，筛选出度数≥1的活跃节点

In [ ]:
time_start=time.time()
# Flatten the array and count the frequency of each node (this gives the degree of each node)
all_nodes_2016, degrees_2016 = np.unique(pairs_2016.values.flatten(), return_counts=True)

# Create a mask for nodes with a degree greater than the cutoff
large_degree_mask = degrees_2016 >= vertex_degree_cutoff
# Get the nodes with a degree greater than the cutoff
vertex_large_degs_2016 = all_nodes_2016[large_degree_mask]
print(f"vertex_used_2016: {len(vertex_large_degs_2016)}; elapsed_time: {time.time()-time_start}")


time_start=time.time()
all_nodes_2019, degrees_2019 = np.unique(pairs_2019.values.flatten(), return_counts=True)
large_degree_mask = degrees_2019 >= vertex_degree_cutoff
vertex_large_degs_2019 = all_nodes_2019[large_degree_mask]
print(f"vertex_used_2019: {len(vertex_large_degs_2019)}; elapsed_time: {time.time()-time_start}")


time_start=time.time()
all_nodes_2022, degrees_2022 = np.unique(pairs_2022.values.flatten(), return_counts=True)
large_degree_mask = degrees_2022 >= vertex_degree_cutoff
vertex_large_degs_2022 = all_nodes_2022[large_degree_mask]
print(f"vertex_used_2022: {len(vertex_large_degs_2022)}; elapsed_time: {time.time()-time_start}")

#### get all the combination of the used vertex

对筛选出的活跃节点，生成所有可能的节点对组合（组合数C(n,2)）

数据量巨大：2016年约1.86亿对，2022年约1.97亿对

In [ ]:
 
time_start=time.time()
n = len(vertex_large_degs_2016)
c, r = np.triu_indices(n, k=1)  # Gets the upper triangle indices excluding the diagonal
combine_pairs_2016 = np.column_stack((vertex_large_degs_2016[c], vertex_large_degs_2016[r]))
print(f"all combine_pairs_2016: {len(combine_pairs_2016)}; elapsed_time: {time.time()-time_start}")

time_start=time.time()
n = len(vertex_large_degs_2019)
c, r = np.triu_indices(n, k=1)  # Gets the upper triangle indices excluding the diagonal
combine_pairs_2019 = np.column_stack((vertex_large_degs_2019[c], vertex_large_degs_2019[r]))
print(f"all combine_pairs_2019: {len(combine_pairs_2019)}; elapsed_time: {time.time()-time_start}")


time_start=time.time()
n = len(vertex_large_degs_2022)
c, r = np.triu_indices(n, k=1)  # Gets the upper triangle indices excluding the diagonal
combine_pairs_2022 = np.column_stack((vertex_large_degs_2022[c], vertex_large_degs_2022[r]))
print(f"all combine_pairs_2022: {len(combine_pairs_2022)}; elapsed_time: {time.time()-time_start}")

将生成的组合保存为pickle文件

In [ ]:
import pickle

# 保存 combine_pairs_2016
with open("combine_pairs_2016.pkl", "wb") as f:
    pickle.dump(combine_pairs_2016, f, protocol=4)

# 保存 combine_pairs_2019
with open("combine_pairs_2019.pkl", "wb") as f:
    pickle.dump(combine_pairs_2019, f, protocol=4)

# 保存 combine_pairs_2022
with open("combine_pairs_2022.pkl", "wb") as f:
    pickle.dump(combine_pairs_2022, f, protocol=4)

print("✅ combine_pairs_2016 / 2019 / 2022 已保存完成")


从pickle文件重新加载组合数据

In [ ]:
import pickle

with open("combine_pairs_2016.pkl", "rb") as f:
    combine_pairs_2016 = pickle.load(f)

with open("combine_pairs_2019.pkl", "rb") as f:
    combine_pairs_2019 = pickle.load(f)

with open("combine_pairs_2022.pkl", "rb") as f:
    combine_pairs_2022 = pickle.load(f)

print("✅ combine_pairs_2016 / 2019 / 2022 已成功加载")


将numpy数组转换为pandas DataFrame格式

In [ ]:
# Convert numpy arrays to pandas DataFrames
time_start=time.time()
all_combine_pairs_2016 = pd.DataFrame(combine_pairs_2016, columns=['v1', 'v2'])
print(f"Convert combine_pairs_2016: {len(all_combine_pairs_2016)}, elapsed_time: {time.time()-time_start}")

time_start=time.time()
all_combine_pairs_2019 = pd.DataFrame(combine_pairs_2019, columns=['v1', 'v2'])
print(f"Convert combine_pairs_2019: {len(all_combine_pairs_2019)}, elapsed_time: {time.time()-time_start}")

time_start=time.time()
all_combine_pairs_2022 = pd.DataFrame(combine_pairs_2022, columns=['v1', 'v2'])
print(f"Convert combine_pairs_2022: {len(all_combine_pairs_2022)}, elapsed_time: {time.time()-time_start}")

### prepare unconnected_pairs

#### unconnected pairs in 2016 准备2016年的未连接对
关键操作：从所有可能组合(1.86亿)中，剔除实际存在的边(85万)

得到约1.85亿个2016年时未连接的节点对

使用分批处理避免内存溢出

In [ ]:
import pandas as pd
import time
import gc
import os

time_start = time.time()

print("构建已连接对的查找集合...")
# 使用set进行快速查找
connected_pairs_set = set(zip(pairs_2016['v1'], pairs_2016['v2']))
print(f"已连接对集合大小: {len(connected_pairs_set):,}")

# 设置批次大小（根据内存情况调整）
BATCH_SIZE = 2_000_000
total_rows = len(all_combine_pairs_2016)
num_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

print(f"\n开始分批处理，总行数: {total_rows:,}，分 {num_batches} 批处理")

# 用于存储所有未连接对的列表
all_unconnected_chunks = []
total_unconnected = 0

for i in range(num_batches):
    batch_start = i * BATCH_SIZE
    batch_end = min((i + 1) * BATCH_SIZE, total_rows)
    
    print(f"▶ 处理第 {i+1}/{num_batches} 批: 行 {batch_start:,} - {batch_end:,}")
    
    # 提取当前批次
    batch_df = all_combine_pairs_2016.iloc[batch_start:batch_end].copy()
    
    # 使用向量化操作检查
    is_connected = pd.Series(
        [(v1, v2) in connected_pairs_set for v1, v2 in zip(batch_df['v1'], batch_df['v2'])],
        index=batch_df.index
    )
    
    # 获取未连接的对
    unconnected_batch = batch_df[~is_connected]
    
    # 如果本批有未连接对，添加到列表
    if len(unconnected_batch) > 0:
        # 重置索引以避免后续合并时的索引冲突
        unconnected_batch = unconnected_batch.reset_index(drop=True)
        all_unconnected_chunks.append(unconnected_batch)
        total_unconnected += len(unconnected_batch)
        print(f"  找到未连接对: {len(unconnected_batch):,} 行 (累计: {total_unconnected:,})")
    else:
        print(f"  本批没有未连接对")
    
    # 清理内存
    del batch_df, is_connected, unconnected_batch
    gc.collect()

print(f"\n所有批次处理完成，开始合并结果...")
print(f"总未连接对: {total_unconnected:,}")

# 方法1：直接合并（如果内存充足）
if total_unconnected > 0:
    print("正在合并所有未连接对...")
    
    # 分批合并，避免一次性加载过多数据
    if len(all_unconnected_chunks) > 10:
        print(f"有 {len(all_unconnected_chunks)} 个数据块，使用分批合并策略")
        
        # 每10个数据块合并一次
        merged_chunks = []
        for j in range(0, len(all_unconnected_chunks), 10):
            end_idx = min(j + 10, len(all_unconnected_chunks))
            batch_to_merge = all_unconnected_chunks[j:end_idx]
            
            print(f"  合并批次 {j//10 + 1}: {len(batch_to_merge)} 个数据块")
            merged_batch = pd.concat(batch_to_merge, ignore_index=True)
            merged_chunks.append(merged_batch)
            
            # 清理内存
            del batch_to_merge
            gc.collect()
        
        # 最终合并
        print("进行最终合并...")
        unconnected_pairs_2016 = pd.concat(merged_chunks, ignore_index=True)
        
    else:
        # 数据块较少，直接合并
        unconnected_pairs_2016 = pd.concat(all_unconnected_chunks, ignore_index=True)
    
    print(f"✅ 合并完成! 总行数: {len(unconnected_pairs_2016):,}")
    
    # 保存到文件
    output_file = "unconnected_pairs_2016.parquet"
    print(f"正在保存到 {output_file}...")
    
    # 使用压缩以减小文件大小
    unconnected_pairs_2016.to_parquet(output_file, compression='snappy')
    print(f"✅ 已保存到 {output_file}")
    
    # 显示一些统计信息
    print(f"\n统计信息:")
    print(f"  总行数: {len(unconnected_pairs_2016):,}")
    print(f"  列名: {list(unconnected_pairs_2016.columns)}")
    print(f"  前几行数据:")
    print(unconnected_pairs_2016.head())
    
else:
    print("没有找到任何未连接对")
    unconnected_pairs_2016 = pd.DataFrame(columns=['v1', 'v2'])

print(f"\n✅ 处理完成!")
print(f"总耗时: {time.time()-time_start:.2f} 秒")

# 清理内存
del all_unconnected_chunks
gc.collect()

将结果保存为parquet文件并释放内存

In [ ]:
import pandas as pd

# 分块写入 Parquet 格式（更快、更节省内存）
unconnected_pairs_2016.to_parquet("unconnected_pairs_2016.parquet", compression='snappy')

# 删除变量释放内存
del unconnected_pairs_2016
import gc; gc.collect()

print("✅ 已保存为 Parquet 文件并释放内存。")


In [ ]:
import pandas as pd
unconnected_pairs_2016 = pd.read_parquet("unconnected_pairs_2016.parquet")


#### check unconnected pairs in 2016 for 2019 (unconnected+citation and connected+citation)
检查2016年的未连接对在2019年的状态

In [ ]:
### in unconnected_pair_2016 but not in pairs_2019
### unconnected pairs keep unconnected in 2019
import time
import pandas as pd
time_start=time.time()

unconnected_pair_2016_2019 = pd.merge(unconnected_pairs_2016, pairs_2019, on=['v1', 'v2'], how='outer', indicator=True)
unconnected_pair_2016_2019 = unconnected_pair_2016_2019[unconnected_pair_2016_2019['_merge'] == 'left_only']
unconnected_pair_2016_2019 = unconnected_pair_2016_2019.drop(columns=['_merge'])

print(f"unconnected_pair_2016_2019: {len(unconnected_pair_2016_2019)}; elapsed_time: {time.time()-time_start}")

### in unconnected_pair_2016 but also in pairs_2019
### unconnected pairs becomes connected in 2019
time_start=time.time()
connected_pair_2016_2019 = pd.merge(pairs_2019,unconnected_pairs_2016, on=['v1', 'v2'], how='inner', indicator=True)
connected_pair_2016_2019 = connected_pair_2016_2019[connected_pair_2016_2019['_merge'] == 'both']
connected_pair_2016_2019 = connected_pair_2016_2019.drop(columns=['_merge'])

print(f"connected_pair_2016_2019: {len(connected_pair_2016_2019)}; elapsed_time: {time.time()-time_start}\n")
print(f"train 2016-2019: total- {len(unconnected_pairs_2016)}; connected-- {len(connected_pair_2016_2019)}; unconnected--{len(unconnected_pair_2016_2019)}")

将1.85亿条未连接对分批(50万/批)与2019年的边进行merge

In [ ]:
##################################################################################################
# 分批处理 unconnected_pairs_2016 与 pairs_2019 的合并，避免内存溢出
##################################################################################################
import gc
BATCH_SIZE = 500000  # 每批处理 50万行，可根据内存情况调整
output_folder = "batch_results"
os.makedirs(output_folder, exist_ok=True)

time_start = time.time()
total_rows = len(unconnected_pairs_2016)
num_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

print(f"共 {total_rows} 行，计划分为 {num_batches} 批处理...\n")

for i in range(num_batches):
    batch_start = i * BATCH_SIZE
    batch_end = min((i + 1) * BATCH_SIZE, total_rows)
    print(f"▶ 正在处理第 {i + 1}/{num_batches} 批: 行 {batch_start} - {batch_end}")

    # 提取当前批次
    batch_df = unconnected_pairs_2016.iloc[batch_start:batch_end].copy()

    # ✅ 1. 找出依然未连接的（left_only）
    batch_unconnected = pd.merge(batch_df, pairs_2019, on=['v1', 'v2'], how='outer', indicator=True)
    batch_unconnected = batch_unconnected[batch_unconnected['_merge'] == 'left_only'].drop(columns=['_merge'])

    # ✅ 2. 找出新连接上的（both）
    batch_connected = pd.merge(pairs_2019, batch_df, on=['v1', 'v2'], how='inner', indicator=True)
    batch_connected = batch_connected[batch_connected['_merge'] == 'both'].drop(columns=['_merge'])

    # 保存
    unconnected_path = os.path.join(output_folder, f"unconnected_pair_2016_2019_part{i+1}.parquet")
    connected_path = os.path.join(output_folder, f"connected_pair_2016_2019_part{i+1}.parquet")

    batch_unconnected.to_parquet(unconnected_path, compression="snappy")
    batch_connected.to_parquet(connected_path, compression="snappy")

    print(f"  已保存：未连接 {len(batch_unconnected)} 行；已连接 {len(batch_connected)} 行")

    # 清理内存
    del batch_df, batch_unconnected, batch_connected
    gc.collect()

print(f"\n✅ 所有批次处理完成，总耗时 {time.time() - time_start:.2f} 秒")

##################################################################################################
# （可选）合并所有批次结果
##################################################################################################

def merge_all_parts(prefix, output_name):
    print(f"\n正在合并所有 {prefix} 分片文件...")
    all_files = sorted([f for f in os.listdir(output_folder) if f.startswith(prefix)])
    dfs = []
    for f in all_files:
        df = pd.read_parquet(os.path.join(output_folder, f))
        dfs.append(df)
    merged = pd.concat(dfs, ignore_index=True)
    merged.to_parquet(output_name, compression="snappy")
    print(f"✅ 已合并保存为 {output_name}，总行数: {len(merged)}")
    del dfs, merged
    gc.collect()

# 如内存允许，可执行以下两行完成最终整合：
# merge_all_parts("connected_pair_2016_2019_part", "connected_pair_2016_2019.parquet")
# merge_all_parts("unconnected_pair_2016_2019_part", "unconnected_pair_2016_2019.parquet")


### 合并 得到connected_pair_2016_2019.parquet和unconnected_pair_2016_2019.parquet

In [ ]:
import pandas as pd
import os
import gc
import time

def merge_large_files_incrementally(input_folder, file_prefix, output_file, batch_size=5):
    """
    增量合并大型分片文件，避免内存溢出
    
    Parameters:
    -----------
    input_folder : str - 分片文件所在文件夹
    file_prefix : str - 分片文件前缀（如 "connected_pair_2016_2019_part"）
    output_file : str - 最终输出文件名
    batch_size : int - 每次合并的文件数量，根据内存情况调整（默认5）
    """
    print(f"\n开始增量合并文件: {file_prefix}")
    print(f"输出文件: {output_file}")
    print(f"每次合并 {batch_size} 个文件\n")
    
    # 获取所有匹配的文件并按顺序排序
    all_files = sorted([f for f in os.listdir(input_folder) 
                        if f.startswith(file_prefix) and f.endswith('.parquet')])
    
    if not all_files:
        print(f"错误: 在 {input_folder} 中没有找到以 {file_prefix} 开头的parquet文件")
        return
    
    total_files = len(all_files)
    print(f"找到 {total_files} 个分片文件")
    
    # 创建临时文件夹存放中间结果
    temp_folder = os.path.join(input_folder, "temp_merge")
    os.makedirs(temp_folder, exist_ok=True)
    
    # 第一阶段：分批合并成中间文件
    intermediate_files = []
    num_batches = (total_files + batch_size - 1) // batch_size
    
    print(f"\n第一阶段：合并为 {num_batches} 个中间文件")
    
    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, total_files)
        batch_files = all_files[start_idx:end_idx]
        
        print(f"\n▶ 合并批次 {batch_idx + 1}/{num_batches}: 处理 {len(batch_files)} 个文件")
        
        # 读取当前批次的所有文件
        dfs = []
        batch_rows = 0
        for i, file_name in enumerate(batch_files):
            file_path = os.path.join(input_folder, file_name)
            print(f"  读取文件 {i+1}/{len(batch_files)}: {file_name}")
            
            try:
                df = pd.read_parquet(file_path)
                rows = len(df)
                batch_rows += rows
                dfs.append(df)
                print(f"    行数: {rows:,}")
            except Exception as e:
                print(f"    读取失败: {e}")
                continue
        
        if not dfs:
            print("  本批次没有成功读取的文件，跳过")
            continue
        
        # 合并当前批次
        print(f"  合并本批次 {len(dfs)} 个文件，总行数: {batch_rows:,}")
        merged_batch = pd.concat(dfs, ignore_index=True)
        
        # 保存中间文件
        temp_file = os.path.join(temp_folder, f"temp_batch_{batch_idx+1}.parquet")
        merged_batch.to_parquet(temp_file, compression='snappy')
        intermediate_files.append(temp_file)
        
        print(f"  已保存中间文件: {temp_file}")
        print(f"  中间文件行数: {len(merged_batch):,}")
        
        # 清理内存
        del dfs, merged_batch
        gc.collect()
    
    # 第二阶段：合并所有中间文件
    print(f"\n第二阶段：合并 {len(intermediate_files)} 个中间文件")
    
    if len(intermediate_files) == 1:
        # 只有一个中间文件，直接重命名
        print("只有一个中间文件，直接作为最终结果")
        os.rename(intermediate_files[0], output_file)
        final_rows = pd.read_parquet(output_file).shape[0]
    else:
        # 分批合并中间文件
        final_dfs = []
        total_final_rows = 0
        
        # 如果中间文件太多，再分批合并
        if len(intermediate_files) > batch_size:
            print("中间文件较多，再次分批合并...")
            
            # 再次合并中间文件
            second_level_files = []
            for i in range(0, len(intermediate_files), batch_size):
                batch = intermediate_files[i:i+batch_size]
                print(f"  合并中间批次 {i//batch_size + 1}: {len(batch)} 个文件")
                
                batch_dfs = []
                for f in batch:
                    df = pd.read_parquet(f)
                    batch_dfs.append(df)
                    os.remove(f)  # 删除已读取的临时文件
                
                merged = pd.concat(batch_dfs, ignore_index=True)
                temp_second = os.path.join(temp_folder, f"temp_second_{i//batch_size+1}.parquet")
                merged.to_parquet(temp_second, compression='snappy')
                second_level_files.append(temp_second)
                
                del batch_dfs, merged
                gc.collect()
            
            intermediate_files = second_level_files
        
        # 最后一次合并
        print(f"最终合并 {len(intermediate_files)} 个文件...")
        for f in intermediate_files:
            df = pd.read_parquet(f)
            final_dfs.append(df)
            total_final_rows += len(df)
            os.remove(f)  # 删除临时文件
        
        # 合并最终结果
        print(f"合并最终结果，总行数: {total_final_rows:,}")
        final_result = pd.concat(final_dfs, ignore_index=True)
        final_result.to_parquet(output_file, compression='snappy')
        final_rows = len(final_result)
        
        del final_dfs, final_result
        gc.collect()
    
    # 清理临时文件夹
    try:
        os.rmdir(temp_folder)
    except:
        print(f"临时文件夹 {temp_folder} 不为空，请手动清理")
    
    print(f"\n✅ 合并完成！")
    print(f"最终文件: {output_file}")
    print(f"总行数: {final_rows:,}")
    print(f"文件大小: {os.path.getsize(output_file) / (1024**3):.2f} GB")

# 使用示例
if __name__ == "__main__":
    input_folder = "batch_results"  # 分片文件所在文件夹
    
    # 根据您的内存情况调整 batch_size
    # - 内存大（32GB+）: batch_size = 10-20
    # - 内存中等（16GB）: batch_size = 5-10  
    # - 内存小（8GB以下）: batch_size = 2-3
    BATCH_SIZE = 5  # 每次合并5个文件
    
    # 合并新连接的配对
    merge_large_files_incrementally(
        input_folder=input_folder,
        file_prefix="connected_pair_2016_2019_part",
        output_file="connected_pair_2016_2019.parquet",
        batch_size=BATCH_SIZE
    )
    
    # 清理内存
    gc.collect()
    
    # 合并未连接的配对（如果内存不够，可以单独运行这个）
    # 注意：未连接的文件更大，可能需要更小的 batch_size
    merge_large_files_incrementally(
        input_folder=input_folder,
        file_prefix="unconnected_pair_2016_2019_part", 
        output_file="unconnected_pair_2016_2019.parquet",
        batch_size=2  # 未连接文件较大，用更小的批次
    )

In [ ]:
import pandas as pd

connected_pair_2016_2019 = pd.read_parquet("connected_pair_2016_2019.parquet")
unconnected_pair_2016_2019 = pd.read_parquet("unconnected_pair_2016_2019.parquet")

print(f"connected_pair_2016_2019: {len(connected_pair_2016_2019)}, unconnected_pair_2016_2019: {len(unconnected_pair_2016_2019)}")


#### unconnected pairs in 2019
类似2016年的流程，从2019年的所有可能组合(1.93亿)中剔除实际存在的边(92万)

得到约1.93亿个2019年未连接对

分批处理并保存

In [ ]:
import time
import pandas as pd
import gc
import os

time_start = time.time()

# 设置批次大小（根据内存情况调整，建议 100万-200万行）
BATCH_SIZE = 250000
total_rows = len(all_combine_pairs_2019)
num_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

print(f"总行数: {total_rows:,}，分为 {num_batches} 批处理")

# 创建临时文件夹存储分批结果
temp_folder = "temp_unconnected_2019"
os.makedirs(temp_folder, exist_ok=True)

# 分批处理
for i in range(num_batches):
    batch_start = i * BATCH_SIZE
    batch_end = min((i + 1) * BATCH_SIZE, total_rows)
    
    print(f"\n▶ 处理第 {i+1}/{num_batches} 批: 行 {batch_start:,} - {batch_end:,}")
    
    # 提取当前批次
    batch_df = all_combine_pairs_2019.iloc[batch_start:batch_end].copy()
    
    # 对当前批次进行 merge
    batch_merged = pd.merge(batch_df, pairs_2019, on=['v1', 'v2'], how='outer', indicator=True)
    batch_unconnected = batch_merged[batch_merged['_merge'] == 'left_only'].drop(columns=['_merge'])
    
    # 保存当前批次结果
    temp_file = os.path.join(temp_folder, f"batch_{i+1}.parquet")
    batch_unconnected.to_parquet(temp_file, compression='snappy')
    
    print(f"  本批找到未连接对: {len(batch_unconnected):,} 行")
    
    # 清理内存
    del batch_df, batch_merged, batch_unconnected
    gc.collect()

print(f"\n✅ 所有批次处理完成，开始合并结果...")

# 合并所有批次结果
all_files = sorted([f for f in os.listdir(temp_folder) if f.startswith('batch_')])
dfs = []
total_unconnected = 0

for f in all_files:
    df = pd.read_parquet(os.path.join(temp_folder, f))
    total_unconnected += len(df)
    dfs.append(df)
    print(f"读取 {f}: {len(df):,} 行 (累计: {total_unconnected:,})")

# 最终合并
unconnected_pairs_2019 = pd.concat(dfs, ignore_index=True)
print(f"\n最终 unconnected_pairs_2019: {len(unconnected_pairs_2019):,} 行")

# 保存最终结果
unconnected_pairs_2019.to_parquet("unconnected_pairs_2019.parquet", compression='snappy')
print(f"已保存到 unconnected_pairs_2019.parquet")

# 清理临时文件
import shutil
shutil.rmtree(temp_folder)

print(f"\n总耗时: {time.time()-time_start:.2f} 秒")

In [ ]:
import pandas as pd

# # 分块写入 Parquet 格式（更快、更节省内存）
unconnected_pairs_2019.to_parquet("unconnected_pairs_2019.parquet", compression='snappy')

# # # 删除变量释放内存
#del unconnected_pairs_2016
import gc; gc.collect()

print("✅ 已保存为 Parquet 文件并释放内存。")

In [ ]:
import pandas as pd
unconnected_pairs_2019 = pd.read_parquet("unconnected_pairs_2019.parquet")

#### check unconnected pairs in 2019 for 2022 (unconnected+citation and connected+citation)

In [ ]:
# time_start=time.time()
# unconnected_pair_2019_2022 = pd.merge(unconnected_pairs_2019, pairs_2022, on=['v1', 'v2'], how='outer', indicator=True)
# unconnected_pair_2019_2022 = unconnected_pair_2019_2022[unconnected_pair_2019_2022['_merge'] == 'left_only']
# unconnected_pair_2019_2022 = unconnected_pair_2019_2022.drop(columns=['_merge'])
# print(f"unconnected_pair_2019_2022: {len(unconnected_pair_2019_2022)}; elapsed_time: {time.time()-time_start}")


# time_start=time.time()
# connected_pair_2019_2022 = pd.merge(pairs_2022, unconnected_pairs_2019, on=['v1', 'v2'], how='inner', indicator=True)
# connected_pair_2019_2022 = connected_pair_2019_2022[connected_pair_2019_2022['_merge'] == 'both']
# connected_pair_2019_2022 = connected_pair_2019_2022.drop(columns=['_merge'])
# print(f"connected_pair_2019_2022, {len(connected_pair_2019_2022)}; elapsed_time: {time.time()-time_start}")
# print(f"eval 2019-2022: total- {len(unconnected_pairs_2019)}; connected-- {len(connected_pair_2019_2022)}; unconnected--{len(unconnected_pair_2019_2022)}")

将1.93亿未连接对分批(200万/批)与2022年的边进行merge

结果：

保持未连接的：约1.93亿

新建立连接的：约1.9万

In [ ]:
##################################################################################################
# 分批处理 unconnected_pairs_2019 与 pairs_2022 的合并，避免内存溢出
##################################################################################################
import gc
BATCH_SIZE = 2_000_000  # 每批 200万行，可根据内存大小调整
output_folder = "batch_results_2019_2022"
os.makedirs(output_folder, exist_ok=True)

time_start = time.time()
total_rows = len(unconnected_pairs_2019)
num_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

print(f"\n共 {total_rows} 行，计划分为 {num_batches} 批处理 (2019→2022)...\n")

for i in range(num_batches):
    batch_start = i * BATCH_SIZE
    batch_end = min((i + 1) * BATCH_SIZE, total_rows)
    print(f"▶ 正在处理第 {i + 1}/{num_batches} 批: 行 {batch_start} - {batch_end}")

    # 当前批次
    batch_df = unconnected_pairs_2019.iloc[batch_start:batch_end].copy()

    # ✅ 1. 依然未连接（left_only）
    batch_unconnected = pd.merge(batch_df, pairs_2022, on=['v1', 'v2'], how='outer', indicator=True)
    batch_unconnected = batch_unconnected[batch_unconnected['_merge'] == 'left_only'].drop(columns=['_merge'])

    # ✅ 2. 新连接上的（both）
    batch_connected = pd.merge(pairs_2022, batch_df, on=['v1', 'v2'], how='inner', indicator=True)
    batch_connected = batch_connected[batch_connected['_merge'] == 'both'].drop(columns=['_merge'])

    # 保存 parquet
    unconnected_path = os.path.join(output_folder, f"unconnected_pair_2019_2022_part{i+1}.parquet")
    connected_path = os.path.join(output_folder, f"connected_pair_2019_2022_part{i+1}.parquet")

    batch_unconnected.to_parquet(unconnected_path, compression="snappy")
    batch_connected.to_parquet(connected_path, compression="snappy")

    print(f"  已保存：未连接 {len(batch_unconnected)} 行；已连接 {len(batch_connected)} 行")

    # 清理内存
    del batch_df, batch_unconnected, batch_connected
    gc.collect()

print(f"\n✅ 所有批次处理完成，总耗时 {time.time() - time_start:.2f} 秒")

##################################################################################################
# （可选）合并所有批次结果
##################################################################################################

def merge_all_parts(prefix, output_name, folder):
    print(f"\n正在合并所有 {prefix} 分片文件...")
    all_files = sorted([f for f in os.listdir(folder) if f.startswith(prefix)])
    dfs = []
    for f in all_files:
        df = pd.read_parquet(os.path.join(folder, f))
        dfs.append(df)
    merged = pd.concat(dfs, ignore_index=True)
    merged.to_parquet(output_name, compression="snappy")
    print(f"✅ 已合并保存为 {output_name}，总行数: {len(merged)}")
    del dfs, merged
    gc.collect()

In [ ]:
##################################################################################################
# 极致内存安全的合并方式（分块读取+分块写入）
# 用于合并 connected_pair_2019_2022_part 和 unconnected_pair_2019_2022_part 分片文件
##################################################################################################
import pandas as pd
import os
import gc
import time
import glob

def merge_all_parts_ultra_safe(prefix, output_name, folder):
    """
    极致内存安全的合并方式，逐个文件处理，避免内存溢出
    
    Parameters:
    - prefix: 文件前缀（如 "connected_pair_2019_2022_part"）
    - output_name: 输出文件名（如 "connected_pair_2019_2022.parquet"）
    - folder: 分片文件所在文件夹（如 "batch_results_2019_2022"）
    """
    time_start = time.time()
    
    # 获取所有分片文件
    pattern = os.path.join(folder, f"{prefix}*.parquet")
    all_files = sorted(glob.glob(pattern))
    
    print(f"\n正在合并 {prefix} 分片文件...")
    print(f"找到 {len(all_files)} 个分片文件")
    
    if not all_files:
        print("没有找到分片文件")
        return
    
    # 读取第一个文件并保存
    print(f"处理文件 1/{len(all_files)}: {os.path.basename(all_files[0])}")
    df_first = pd.read_parquet(all_files[0])
    df_first.to_parquet(output_name, compression='snappy', index=False)
    total_rows = len(df_first)
    print(f"  创建文件，写入 {len(df_first):,} 行")
    
    # 释放内存
    del df_first
    gc.collect()
    
    # 逐个处理剩余的文件
    for i, file_path in enumerate(all_files[1:], start=2):
        print(f"处理文件 {i}/{len(all_files)}: {os.path.basename(file_path)}")
        
        # 读取当前分片文件
        df = pd.read_parquet(file_path)
        
        # 读取现有文件并追加
        df_existing = pd.read_parquet(output_name)
        df_combined = pd.concat([df_existing, df], ignore_index=True)
        df_combined.to_parquet(output_name, compression='snappy', index=False)
        
        total_rows = len(df_combined)
        print(f"  追加写入 {len(df):,} 行，累计: {total_rows:,} 行")
        
        # 释放内存
        del df, df_existing, df_combined
        gc.collect()
    
    print(f"\n✅ 合并完成！总行数: {total_rows:,}")
    print(f"  保存为: {output_name}")
    print(f"  耗时: {time.time()-time_start:.2f} 秒")

##################################################################################################
# 更高效的版本：使用临时文件避免重复读取整个文件
##################################################################################################
def merge_all_parts_efficient(prefix, output_name, folder, temp_suffix='.tmp'):
    """
    更高效的合并方式，使用临时文件避免重复读取整个文件
    
    Parameters:
    - prefix: 文件前缀
    - output_name: 输出文件名
    - folder: 分片文件所在文件夹
    - temp_suffix: 临时文件后缀
    """
    time_start = time.time()
    
    # 获取所有分片文件
    pattern = os.path.join(folder, f"{prefix}*.parquet")
    all_files = sorted(glob.glob(pattern))
    
    print(f"\n正在合并 {prefix} 分片文件...")
    print(f"找到 {len(all_files)} 个分片文件")
    
    if not all_files:
        print("没有找到分片文件")
        return
    
    # 创建临时文件名
    temp_file = output_name + temp_suffix
    
    # 复制第一个文件作为基础
    print(f"处理文件 1/{len(all_files)}: {os.path.basename(all_files[0])}")
    df_first = pd.read_parquet(all_files[0])
    df_first.to_parquet(temp_file, compression='snappy', index=False)
    total_rows = len(df_first)
    print(f"  基础文件创建完成，{len(df_first):,} 行")
    
    del df_first
    gc.collect()
    
    # 逐个追加剩余文件
    for i, file_path in enumerate(all_files[1:], start=2):
        print(f"处理文件 {i}/{len(all_files)}: {os.path.basename(file_path)}")
        
        # 读取当前文件
        df_current = pd.read_parquet(file_path)
        
        # 读取临时文件
        df_temp = pd.read_parquet(temp_file)
        
        # 合并
        df_combined = pd.concat([df_temp, df_current], ignore_index=True)
        df_combined.to_parquet(temp_file, compression='snappy', index=False)
        
        total_rows = len(df_combined)
        print(f"  追加完成，累计: {total_rows:,} 行")
        
        # 释放内存
        del df_current, df_temp, df_combined
        gc.collect()
    
    # 重命名临时文件为最终文件
    if os.path.exists(output_name):
        os.remove(output_name)
    os.rename(temp_file, output_name)
    
    print(f"\n✅ 合并完成！总行数: {total_rows:,}")
    print(f"  保存为: {output_name}")
    print(f"  耗时: {time.time()-time_start:.2f} 秒")

##################################################################################################
# 最简单的方法：使用命令行工具（如果数据量极大）
##################################################################################################
def merge_parquet_files_cli(prefix, output_name, folder):
    """
    使用 pyarrow 或 pandas 的 concat 一次性合并（如果内存足够）
    或者建议使用命令行工具如 parquet-tools
    """
    import subprocess
    
    print(f"建议使用 parquet-tools 命令行工具合并:")
    print(f"parquet-tools merge {folder}/{prefix}* {output_name}")
    
    # 或者使用 pyarrow 的 parquet 写入器（需要安装 pyarrow）
    try:
        import pyarrow.parquet as pq
        import pyarrow as pa
        
        pattern = os.path.join(folder, f"{prefix}*.parquet")
        all_files = sorted(glob.glob(pattern))
        
        writer = None
        total_rows = 0
        
        for file_path in all_files:
            table = pq.read_table(file_path)
            if writer is None:
                writer = pq.ParquetWriter(output_name, table.schema, compression='snappy')
            writer.write_table(table)
            total_rows += table.num_rows
            print(f"写入 {os.path.basename(file_path)}: {table.num_rows:,} 行")
        
        if writer:
            writer.close()
        
        print(f"\n✅ 合并完成！总行数: {total_rows:,}")
        
    except ImportError:
        print("请安装 pyarrow: pip install pyarrow")

##################################################################################################
# 主程序：执行合并操作
##################################################################################################

if __name__ == "__main__":
    
    # 定义路径
    input_folder = "batch_results_2019_2022"  # 分片文件所在文件夹
    
    print("="*60)
    print("开始合并 connected 分片文件...")
    print("="*60)
    
    # 使用高效版本合并 connected 分片
    merge_all_parts_efficient(
        prefix="connected_pair_2019_2022_part",           # 输入文件前缀
        output_name="connected_pair_2019_2022.parquet",   # 输出文件
        folder=input_folder                                # 输入文件夹
    )
    
    print("\n" + "="*60)
    print("开始合并 unconnected 分片文件...")
    print("="*60)
    
    # 合并 unconnected 分片
    merge_all_parts_efficient(
        prefix="unconnected_pair_2019_2022_part",         # 输入文件前缀
        output_name="unconnected_pair_2019_2022.parquet", # 输出文件
        folder=input_folder                                # 输入文件夹
    )
    
    print("\n" + "="*60)
    print("✅ 所有合并操作完成！")
    print("="*60)
    
    # 显示合并后的文件信息
    for fname in ["connected_pair_2019_2022.parquet", "unconnected_pair_2019_2022.parquet"]:
        if os.path.exists(fname):
            size = os.path.getsize(fname) / (1024**3)
            rows = len(pd.read_parquet(fname))  # 只读行数，不加载全部数据
            print(f"{fname}: {rows:,} 行, {size:.2f} GB")

In [ ]:
import pandas as pd

# 直接读取保存的 parquet 文件
connected_pair_2019_2022 = pd.read_parquet("connected_pair_2019_2022.parquet")
unconnected_pair_2019_2022 = pd.read_parquet("unconnected_pair_2019_2022.parquet")

# 输出基本信息
print(f"connected_pair_2019_2022: {len(connected_pair_2019_2022)}, unconnected_pair_2019_2022: {len(unconnected_pair_2019_2022)}")


#### unconnected pair in 2022 (no future eval)

In [ ]:
# time_start=time.time()

# unconnected_pairs_2022 = pd.merge(all_combine_pairs_2022, pairs_2022, on=['v1', 'v2'], how='outer', indicator=True)
# unconnected_pairs_2022 = unconnected_pairs_2022[unconnected_pairs_2022['_merge'] == 'left_only']
# unconnected_pairs_2022 = unconnected_pairs_2022.drop(columns=['_merge'])

# print(f"unconnected_pairs_2022: {len(unconnected_pairs_2022)}; elapsed_time: {time.time()-time_start}")


# store_folder="data_pair_solution"
# if not os.path.exists(store_folder):
#     os.makedirs(store_folder)
# print(f"store files in {store_folder}.....")

# ### unconnected pair are connected in 2019, 2022

# time_start = time.time()
# store_name=os.path.join(store_folder,"unconnected_pair_2022.parquet")
# unconnected_pairs_2022.to_parquet(store_name, compression='gzip')
# print(f"unconnected_pairs_2022: {len(unconnected_pairs_2022)}; elapsed_time: {time.time() - time_start}")


In [ ]:
##################################################################################################
# 分批计算 unconnected_pairs_2022 （避免一次性 merge 爆内存）
##################################################################################################

BATCH_SIZE = 2_000_000  # 每批200万行，可根据机器内存调整
store_folder = "data_pair_solution"
os.makedirs(store_folder, exist_ok=True)
output_folder = os.path.join(store_folder, "unconnected_pairs_2022_parts")
os.makedirs(output_folder, exist_ok=True)

time_start = time.time()
total_rows = len(all_combine_pairs_2022)
num_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

print(f"\n共 {total_rows} 行，计划分为 {num_batches} 批处理以生成 unconnected_pairs_2022...\n")

for i in range(num_batches):
    batch_start = i * BATCH_SIZE
    batch_end = min((i + 1) * BATCH_SIZE, total_rows)
    print(f"▶ 正在处理第 {i + 1}/{num_batches} 批: 行 {batch_start} - {batch_end}")

    # 当前批次
    batch_df = all_combine_pairs_2022.iloc[batch_start:batch_end].copy()

    # 找出 unconnected（仅在 all_combine_pairs_2022 中、不在 pairs_2022 中）
    batch_unconnected = pd.merge(batch_df, pairs_2022, on=['v1', 'v2'], how='outer', indicator=True)
    batch_unconnected = batch_unconnected[batch_unconnected['_merge'] == 'left_only'].drop(columns=['_merge'])

    # 保存分批结果
    part_path = os.path.join(output_folder, f"unconnected_pairs_2022_part{i+1}.parquet")
    batch_unconnected.to_parquet(part_path, compression="snappy")

    print(f"  已保存第 {i + 1} 批：未连接 {len(batch_unconnected)} 行")

    # 清理内存
    del batch_df, batch_unconnected
    gc.collect()

print(f"\n✅ 所有批次处理完成，总耗时 {time.time() - time_start:.2f} 秒")

##################################################################################################
# （可选）合并所有批次结果为最终文件
##################################################################################################

def merge_all_parts(prefix, output_name, folder):
    print(f"\n正在合并所有 {prefix} 分片文件...")
    all_files = sorted([f for f in os.listdir(folder) if f.startswith(prefix)])
    dfs = []
    for f in all_files:
        df = pd.read_parquet(os.path.join(folder, f))
        dfs.append(df)
    merged = pd.concat(dfs, ignore_index=True)
    merged.to_parquet(output_name, compression="gzip")
    print(f"✅ 已合并保存为 {output_name}，总行数: {len(merged)}")
    del dfs, merged
    gc.collect()

# 如需生成最终完整文件，请在内存充足时执行：
# final_store_name = os.path.join(store_folder, "unconnected_pair_2022.parquet")
# merge_all_parts("unconnected_pairs_2022_part", final_store_name, output_folder)


In [ ]:
import pandas as pd
import os
import glob
import gc
import time

def merge_unconnected_parts():
    """
    将data_pair_solution目录下的unconnected_pairs_2022_partX.parquet分片文件
    分批次合并成一个unconnected_pair_2022.parquet文件
    """
    print("="*60)
    print("开始合并 unconnected_pairs_2022 分片文件")
    print("="*60)
    
    # 定义路径
    folder_path = "E:\\study\\research\\impact4cast_cscl\\data_pair_solution"
    parts_folder = os.path.join(folder_path, "unconnected_pairs_2022_parts")
    output_file = os.path.join(folder_path, "unconnected_pair_2022.parquet")
    
    # 检查分片文件夹是否存在
    if not os.path.exists(parts_folder):
        print(f"❌ 错误: 分片文件夹不存在 - {parts_folder}")
        return
    
    # 获取所有分片文件
    pattern = os.path.join(parts_folder, "unconnected_pairs_2022_part*.parquet")
    all_files = sorted(glob.glob(pattern))
    
    if not all_files:
        print(f"❌ 错误: 没有找到分片文件")
        return
    
    print(f"找到 {len(all_files)} 个分片文件")
    
    # 如果输出文件已存在，先删除
    if os.path.exists(output_file):
        os.remove(output_file)
        print(f"已删除现有输出文件")
    
    # 合并参数
    BATCH_SIZE = 10  # 每批处理10个文件
    total_batches = (len(all_files) + BATCH_SIZE - 1) // BATCH_SIZE
    total_rows = 0
    start_time = time.time()
    
    print(f"\n开始分批次合并，共 {total_batches} 个批次...\n")
    
    # 分批次处理
    for batch_idx in range(total_batches):
        batch_start = batch_idx * BATCH_SIZE
        batch_end = min((batch_idx + 1) * BATCH_SIZE, len(all_files))
        batch_files = all_files[batch_start:batch_end]
        
        print(f"▶ 处理第 {batch_idx + 1}/{total_batches} 批次 ({len(batch_files)} 个文件)")
        
        # 读取当前批次的所有文件
        batch_dfs = []
        batch_rows = 0
        
        for file_path in batch_files:
            df = pd.read_parquet(file_path)
            file_name = os.path.basename(file_path)
            file_rows = len(df)
            batch_dfs.append(df)
            batch_rows += file_rows
            print(f"  读取 {file_name}: {file_rows:,} 行")
        
        # 合并当前批次
        print(f"  合并当前批次数据 ({batch_rows:,} 行)...")
        batch_merged = pd.concat(batch_dfs, ignore_index=True)
        
        if batch_idx == 0:
            # 第一批次，直接保存
            batch_merged.to_parquet(output_file, compression='snappy', index=False)
            print(f"  创建文件，写入 {len(batch_merged):,} 行")
        else:
            # 后续批次：读取现有文件，合并，重新保存
            existing_df = pd.read_parquet(output_file)
            combined_df = pd.concat([existing_df, batch_merged], ignore_index=True)
            os.remove(output_file)
            combined_df.to_parquet(output_file, compression='snappy', index=False)
            print(f"  追加 {len(batch_merged):,} 行，累计 {len(combined_df):,} 行")
            del existing_df, combined_df
        
        total_rows += len(batch_merged)
        print(f"  当前累计总行数: {total_rows:,} 行\n")
        
        # 清理内存
        del batch_dfs, batch_merged
        gc.collect()
    
    # 计算耗时和文件大小
    elapsed_time = time.time() - start_time
    file_size = os.path.getsize(output_file) / (1024**3)
    
    print("="*60)
    print(f"✅ 合并完成！")
    print(f"  输出文件: {output_file}")
    print(f"  总行数: {total_rows:,} 行")
    print(f"  文件大小: {file_size:.2f} GB")
    print(f"  总耗时: {elapsed_time:.2f} 秒")
    print("="*60)

# 执行合并
if __name__ == "__main__":
    merge_unconnected_parts()

### unconnected pair and solution (citation information); train
正样本：连接上的对(6.3万)，合并2019年的完整图数据获取引用信息
• 负样本：未连接的对(1.85亿)，添加citation=0列

In [ ]:
time_start=time.time()
pair_solution_connected_2019=pd.merge(connected_pair_2016_2019,full_graph_2019, on=['v1', 'v2'], how='inner')
print(f"2016 connected in 2019  : {len(pair_solution_connected_2019)}; elapsed_time: {time.time()-time_start}")

time_start=time.time()
pair_solution_unconnected_2019=unconnected_pair_2016_2019
pair_solution_unconnected_2019.insert(2, 'citation', 0)
print(f"2016 unconnected in 2019: {len(pair_solution_unconnected_2019)}; elapsed_time: {time.time()-time_start}")

同理处理2019→2022的数据：
◦ 正样本：1.9万
◦ 负样本：1.93亿

In [ ]:
time_start=time.time()
pair_solution_connected_2022=pd.merge(connected_pair_2019_2022,full_graph_2022, on=['v1', 'v2'], how='inner')
print(f"2019 connected in 2022  : {len(pair_solution_connected_2022)}; elapsed_time: {time.time()-time_start}")

time_start=time.time()
pair_solution_unconnected_2022=unconnected_pair_2019_2022
pair_solution_unconnected_2022.insert(2, 'citation', 0)
print(f"2019 unconnected in 2022: {len(pair_solution_unconnected_2022)}; elapsed_time: {time.time()-time_start}")


#### store orginal cases
对正样本进行聚合：
◦ 计算每对节点的总引用数（根据时间窗口内的年份求和）
◦ 计算平均引用数(citation_m = citation/出现次数)
◦ 保存最终处理后的训练数据

In [ ]:
store_folder="data_pair_solution"
if not os.path.exists(store_folder):
    os.makedirs(store_folder)
print(f"store files in {store_folder}.....")

### unconnected pair are connected in 2019, 2022

time_start = time.time()
store_name=os.path.join(store_folder,"unconnected_2016_pair_solution_connected_2019_full.parquet")
pair_solution_connected_2019.to_parquet(store_name, compression='gzip')
print(f"pair_solution_connected_2019 full: {len(pair_solution_connected_2019)}; elapsed_time: {time.time() - time_start}")


time_start = time.time()
store_name=os.path.join(store_folder,"unconnected_2019_pair_solution_connected_2022_full.parquet")
pair_solution_connected_2022.to_parquet(store_name, compression='gzip')
print(f"pair_solution_connected_2022 full: {len(pair_solution_connected_2022)}; elapsed_time: {time.time() - time_start}\n")


### unconnected pair are not connected in 2019, 2022

time_start = time.time()
store_name=os.path.join(store_folder,"unconnected_2016_pair_solution_unconnected_2019.parquet")
pair_solution_unconnected_2019.to_parquet(store_name, compression='gzip')
print(f"pair_solution_unconnected_2019: {len(pair_solution_unconnected_2019)}; elapsed_time: {time.time() - time_start}")


time_start = time.time()
store_name=os.path.join(store_folder,"unconnected_2019_pair_solution_unconnected_2022.parquet")
pair_solution_unconnected_2022.to_parquet(store_name, compression='gzip')
print(f"pair_solution_unconnected_2022: {len(pair_solution_unconnected_2022)}; elapsed_time: {time.time() - time_start}")

#### merge repeated pairs 

In [ ]:
time_start = time.time()

# Use .groupby to group by 'v1' and 'v2', then use .sum to get the total citations for each pair
grouped_data_df=pair_solution_connected_2019.copy()
grouped_data_df['citation']=pair_solution_connected_2019[['c2019', 'c2018', 'c2017']].sum(axis=1)
dynamic_grouped_data = grouped_data_df.groupby(['v1','v2']).agg({'citation':'sum','v1':'size'}).rename(columns={'v1':'num'}).reset_index()
dynamic_grouped_data['citation_m'] = dynamic_grouped_data[f'citation'] / dynamic_grouped_data['num']
print(f"elapsed_time: {time.time() - time_start}")

time_start = time.time()
store_folder="data_pair_solution" 
store_name=os.path.join(store_folder,"unconnected_2016_pair_solution_connected_2019.parquet")
dynamic_grouped_data.to_parquet(store_name, compression='gzip')
print(f"unconnected_2016_pair_solution_connected_2019: {len(dynamic_grouped_data)}; elapsed_time: {time.time() - time_start}")

In [ ]:
time_start = time.time()
grouped_data_df=pair_solution_connected_2022.copy()
grouped_data_df['citation']=pair_solution_connected_2022[['c2022', 'c2021', 'c2020']].sum(axis=1)
dynamic_grouped_data = grouped_data_df.groupby(['v1','v2']).agg({'citation':'sum','v1':'size'}).rename(columns={'v1':'num'}).reset_index()
dynamic_grouped_data['citation_m'] = dynamic_grouped_data['citation'] / dynamic_grouped_data['num']
print(f"elapsed_time: {time.time() - time_start}")

time_start = time.time()
store_folder="data_pair_solution" 
store_name=os.path.join(store_folder,"unconnected_2019_pair_solution_connected_2022.parquet")
dynamic_grouped_data.to_parquet(store_name, compression='gzip')
print(f"unconnected_2019_pair_solution_connected_2022_processed: {len(dynamic_grouped_data)}; elapsed_time: {time.time() - time_start}")